# The Heat Equation by Finite Differences

The heat equation $u_t = D\,u_{xx}$ says that $u$ rises where it lies below the average of its neighbours and falls where it lies above. This notebook replaces $u_{xx}$ by a second difference, shows why the obvious explicit scheme blows up, solves the equation properly by the **method of lines**, and checks the result against the exact Fourier solution.

In [ ]:
using OrdinaryDiffEq, SparseArrays, LinearAlgebra, Plots
function laplacian(n; L=1.0)   # interior grid and second-difference matrix, u = 0 at both ends
    h = L / (n + 1)
    return h .* (1:n), spdiagm(-1 => ones(n - 1), 0 => fill(-2.0, n), 1 => ones(n - 1)) ./ h^2
end

## The second difference

$(u_{i-1} - 2u_i + u_{i+1})/h^2$ approximates $u''(x_i)$ with error $O(h^2)$. For $\sin(\pi x)$ the exact second derivative is $-\pi^2\sin(\pi x)$, and each halving of $h$ cuts the error by 4.

In [ ]:
map((9, 19, 39, 79, 159)) do n
    x, A = laplacian(n)
    (n = n, h = 1 / (n + 1), error = maximum(abs.(A * sin.(pi .* x) .+ pi^2 .* sin.(pi .* x))))
end

## An explicit scheme, and its instability

The obvious scheme steps forward in time: $u^{k+1} = u^k + \Delta t\, D A u^k$. It is stable only if $\Delta t \le h^2/(2D)$. Here $h = 0.02$ and $D = 0.01$, so the limit is 0.02. The initial profile is a block, $u = 1$ on $(0.4, 0.6)$.

In [ ]:
D, n = 0.01, 49
xs, A = laplacian(n)
u0 = [0.4 < x < 0.6 ? 1.0 : 0.0 for x in xs]
function explicit(dt, T)
    u = copy(u0)
    for _ in 1:round(Int, T / dt)
        u = u .+ dt .* D .* (A * u)
    end
    return u
end
plot(plot(xs, [u0 explicit(0.019, 2.0)], lw=2, label=["t = 0" "t = 2"], title="dt = 0.019: stable"),
     plot(xs, explicit(0.021, 2.0), lw=2, label="t = 2", title="dt = 0.021: unstable"), layout=(1, 2), size=(900, 330), xlabel="x")

The fastest mode of the grid, a sawtooth, decays by a factor $1 - 4D\Delta t/h^2$ per step. Once that factor falls below $-1$ it grows and flips sign every step. Halving $h$ would quarter the allowed step: explicit schemes become hopeless on fine grids.

## The method of lines

Keep time continuous. The discretised equation $u' = DAu$ is a system of $n$ ODEs, and a stiff one, so an implicit solver such as FBDF takes large, stable, error-controlled steps.

In [ ]:
rhs(u, p, t) = D .* (A * u)
sol = solve(ODEProblem(rhs, u0, (0.0, 5.0)), FBDF(); reltol=1e-6)
plot(xs, [sol(t) for t in (0.0, 0.1, 0.5, 1.0, 2.0, 5.0)], lw=2, label=["t = 0" "t = 0.1" "t = 0.5" "t = 1" "t = 2" "t = 5"], xlabel="x", ylabel="u", title="Diffusion smooths a block")

In [ ]:
lam = eigvals(Matrix(D .* A))
(fastest_rate = maximum(abs.(lam)), slowest_rate = minimum(abs.(lam)), stiffness_ratio = maximum(abs.(lam)) / minimum(abs.(lam)), fbdf_steps = length(sol.t) - 1,
 tsit5_steps = length(solve(ODEProblem(rhs, u0, (0.0, 5.0)), Tsit5(); reltol=1e-6).t) - 1)

## Check against the exact solution

Each sine mode decays on its own: $\sin(k\pi x)$ becomes $\sin(k\pi x)\,e^{-D k^2\pi^2 t}$. So for $u_0 = \sin(\pi x) + \tfrac12\sin(3\pi x)$ the exact solution is known. The error is set by the grid, and it falls by about 4 when $h$ halves, even though the solver tolerance is tight.

In [ ]:
exact(x, t) = sin(pi * x) * exp(-D * pi^2 * t) + 0.5 * sin(3pi * x) * exp(-9D * pi^2 * t)
map((24, 49, 99, 199)) do m
    x, B = laplacian(m)
    s = solve(ODEProblem((u, p, t) -> D .* (B * u), exact.(x, 0.0), (0.0, 2.0)), FBDF(); reltol=1e-10, abstol=1e-12)
    (n = m, h = 1 / (m + 1), error_at_t_2 = maximum(abs.(s.u[end] .- exact.(x, 2.0))))
end

## Boundary values and steady state

Holding the left end at 1 and the right at 0 adds the boundary values through a vector $b$: $u' = D(Au + b)$ with $b_1 = 1/h^2$. The rod settles to the straight line $u = 1 - x$. The slowest mode sets the time scale, $1/(D\pi^2) \approx 10$.

In [ ]:
b = zeros(n); b[1] = 1 / (1 / (n + 1))^2
sol = solve(ODEProblem((u, p, t) -> D .* (A * u .+ b), zeros(n), (0.0, 50.0)), FBDF())
plot(xs, [sol(t) for t in (1.0, 5.0, 10.0, 20.0, 50.0)], lw=2, label=["t = 1" "t = 5" "t = 10" "t = 20" "t = 50"], xlabel="x", title="Heating from the left end")
plot!(xs, 1 .- xs, ls=:dash, c=:black, label="steady state 1 - x")

## Exercises

1. Find the largest stable explicit step for $n = 99$. How many explicit steps would reach $t = 5$?
2. Replace the zero ends by no flux ($A_{11} = A_{nn} = -1/h^2$ on a cell-centred grid). Check that $h\sum u$ stays constant.
3. Implement Crank-Nicolson, $(I - \tfrac{\Delta t}{2}DA)u^{k+1} = (I + \tfrac{\Delta t}{2}DA)u^k$, with `lu` of a sparse matrix. Is it stable for $\Delta t = 0.1$? Does it damp the sawtooth mode?
4. Try the panel's Heat Equation tab with the block initial profile `(0.4 < x < 0.6) * 1.0` and run Grid Convergence. Why is the observed order lower than 2?